# 🚀 LLM Gateway Explained — Build One With LiteLLM + LangChain


---

## we'll cover:

1. **What is an LLM Gateway?** — The problem it solves
2. **Why do we need it?** — Real production pain points
3. **Core capabilities** — Routing, fallbacks, caching, observability, cost tracking
4. **Practical implementation** — Build one from scratch using `LiteLLM`
5. **Integration with LangChain** — Plug the gateway into your agentic apps
6. **Production patterns** — Logging, retries, multi-provider fallbacks

By the end, we'll have a **working LLM gateway** that routes between OpenAI, Anthropic, and Groq — with caching, fallbacks, and cost tracking built in. 🔥

---

## 🧠 Part 1: What is an LLM Gateway?

Think of an **LLM Gateway** as a **smart middleware layer** that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

```
                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
```

### Without a Gateway (The Pain 😩)

- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

### With a Gateway (The Joy 😎)

- **One unified API** for 100+ providers
- **Automatic fallbacks** if a provider fails
- **Centralized logging, cost tracking, rate limiting**
- **Swap models with a config change**, no code rewrite
- **Cache repeated queries** → save money


## ⚙️ Part 2: Installation & Setup

We'll use:
- **LiteLLM** → the most popular open-source LLM gateway (supports 100+ providers)
- **LangChain** → for building agentic workflows on top of the gateway
- **python-dotenv** → for managing API keys

In [12]:
# Install the required packages
!pip install -q litellm langchain langchain-community langchain-openai python-dotenv

In [13]:
#!C:\Python314\python.exe -m pip install --upgrade pip

In [4]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [6]:
import litellm
litellm.suppress_debug_info = True
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [7]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...
# GROQ_API_KEY=gsk_...

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅


## 🎯 Part 3: The Simplest LiteLLM Example — Unified API

The biggest pain point: **every provider has a different SDK**.

LiteLLM gives you **one function** — `completion()` — that works with all of them. Look at how clean this is:

In [8]:
from litellm import completion

# Same code, different providers — just change the `model` string!

# Call OpenAI
response_openai = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🔵 OpenAI:    ", response_openai.choices[0].message.content)



# Call Groq (super fast inference)
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🟢 Groq:      ", response_groq.choices[0].message.content)

🔵 OpenAI:     RAG (Retrieval-Augmented Generation) is a natural language processing approach that combines external knowledge retrieval with generative models to enhance the quality and relevance of generated responses.
🟢 Groq:       RAG (Retrieve, Augment, Generate) is a type of artificial intelligence model that combines retrieval of relevant information from a large database or knowledge source with generation capabilities to produce more accurate and informative responses.


**🎉 Notice:** Same code, three different providers. This alone is huge — you can switch providers with a string change.

But a real LLM Gateway does much more. Let's build those features one by one. 👇

In [9]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : RAG, or Retrieval-Augmented Generation, is a method that combines information re
🟢 Groq         : RAG (Retrieve, Augment, Generate) is a type of artificial intelligence model tha
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ AuthenticationError


## 🛡️ Part 4: Automatic Fallbacks — When OpenAI Goes Down

**Real story:** OpenAI had a 4-hour outage in November 2023. Apps that hard-coded `gpt-4` went completely dark.

With a gateway, if one provider fails, we **automatically fall back** to another. Production apps must have this.

In [10]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.3-70b-versatile"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

17:59:46 - LiteLLM:ERROR: fallback_utils.py:62 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.AuthenticationError: GeminiException - {
  "error": {
    "code": 400,
    "message": "API key not valid. Please pass a valid API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "generativelanguage.googleapis.com"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.LocalizedMessage",
        "locale": "en-US",
        "message": "API key not valid. Please pass a valid API key."
      }
    ]
  }
}
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2033, in async_completion
    response = await client.post(
       

Response: An LLM Gateway typically refers to a system or service that acts as an interface to interact with Large Language Models (LLMs). These gateways facilitate access to and communication with LLMs, enablin ...

Which model actually answered? gpt-4o-mini-2024-07-18


If `gpt-4o-mini` is rate-limited or down, LiteLLM transparently retries with Claude, then Groq. Your app **never sees the failure**.

This is the #1 reason teams adopt an LLM Gateway.

In [11]:
from litellm import completion

# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # 👈 will fail intentionally
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",                              # 1st backup: real OpenAI model
        "groq/llama-3.3-70b-versatile"              # 2nd backup: Groq
    ]
)

print("✅ App still got a response, even though the primary failed!")
print(f"\n🤖 Model that actually answered: {response.model}")
print(f"\n📝 Response: {response.choices[0].message.content[:200]}...")

18:00:31 - LiteLLM:ERROR: fallback_utils.py:62 - Fallback attempt failed for model openai/fake-nonexistent-model-9999: litellm.NotFoundError: OpenAIException - The model `fake-nonexistent-model-9999` does not exist or you do not have access to it.
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 823, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 190, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 454, in make_openai_chat_completion_request
    raise e
  File 

✅ App still got a response, even though the primary failed!

🤖 Model that actually answered: gpt-4o-mini-2024-07-18

📝 Response: An LLM Gateway, or Large Language Model Gateway, typically refers to an interface or a system that allows users to access and interact with large language models (LLMs) like OpenAI's GPT-3 or other si...


## 💰 Part 5: Cost Tracking — Know Where Your Money Goes

LiteLLM **automatically calculates the cost** of every call using its built-in pricing database. No more surprise bills.

In [12]:
from litellm import completion, completion_cost

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Whispers of pure code,  
Learning, adapting, and grow—  
Silent mind aglow.  

Input tokens:  14
Output tokens: 21
Cost:         $0.00001470


## ⚡ Part 6: Caching — Don't Pay Twice for the Same Question

If 100 users ask *"What is RAG?"*, you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [13]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [14]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️  First call (API):   1.15s — LLM stands for "Large Language Model."
⚡ Second call (cache): 0.0040s — LLM stands for "Large Language Model."

🚀 Speedup: 285.8x faster, and ZERO cost on the second call!


## 🔀 Part 7: Smart Routing — The Right Model for the Right Job

**Why use one model for everything?**

- Coding tasks → Claude Sonnet
- Cheap summaries → GPT-4o-mini
- Super fast replies → Groq Llama
- Complex reasoning → Claude Opus

Use LiteLLM's **Router** to define routing rules:

In [15]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "gpt-4o",                                      # 👈 mapped to OpenAI instead
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  The statement "AI is changing software" refers to the significant impact Artificial Intelligence (AI) is having on the software development industry. 

🧠 Smart/coding (GPT-4o):
 Certainly! Here's a simple Python function to reverse a string:

```python
def reverse_string(input_string):
    """
    Reverses the given string.

    Parameters:
    input_string (str): The string to be reversed.

    Returns:
    str: The reversed string.
    """
    return input_string[::-1]

#


## 🔁 Part 8: Load Balancing Across Multiple API Keys

Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [16]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq-llama-70b           575 ms   Hello, I'm happy to assist you with
#2        groq-llama-70b           538 ms   Hello. This is your request 2. How 
#3        groq-llama-70b           629 ms   Hello. You've requested 3, which I 
#4        openai-gpt4o            1737 ms   Hello! How can I assist you today w
#5        groq-llama-70b           700 ms   Hello! You've requested 5, could yo
#6        groq-llama-70b           620 ms   Hello. Your request is not clear, c


### 🎯 Strategy 1: least-busy —
 The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [17]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 OpenAI"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 OpenAI
Request 2 → 🔵 OpenAI
Request 3 → 🔵 OpenAI
Request 4 → 🔵 OpenAI
Request 5 → 🔵 OpenAI
Request 6 → 🔵 OpenAI
Request 7 → 🔵 OpenAI
Request 8 → 🔵 OpenAI

🎯 Distribution:
   🔵 OpenAI: ████████ (8)


### 🎯 Strategy 2: latency-based-routing — 
The "Always Pick the Fastest" Pattern
The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins.

In [33]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 OpenAI GPT-4o-mini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama-3.3"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🟢 Groq Llama-3.3                   486 ms
#2    🟢 Groq Llama-3.3                   669 ms
#3    🔵 OpenAI GPT-4o-mini              1299 ms
#4    🟢 Groq Llama-3.3                   652 ms
#5    🔵 OpenAI GPT-4o-mini               711 ms
#6    🔵 OpenAI GPT-4o-mini              1102 ms
#7    🔵 OpenAI GPT-4o-mini               880 ms
#8    🟢 Groq Llama-3.3                   122 ms
#9    🔵 OpenAI GPT-4o-mini               871 ms
#10   🟢 Groq Llama-3.3                   556 ms


### 🎯 Strategy 4: cost-based-routing — The "Always Cheapest" Pattern
The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

In [28]:
import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o",             # ~$2.50/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o (premium)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",        # ~$0.15/M input tokens
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 GPT-4o-mini (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="cost-based-routing"   # 👈 valid strategy
)

for i in range(5):
    r = await router.acompletion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🔵 GPT-4o-mini (cheap)
Request 2 → 🔵 GPT-4o-mini (cheap)
Request 3 → 🔵 GPT-4o-mini (cheap)
Request 4 → 🔵 GPT-4o-mini (cheap)
Request 5 → 🔵 GPT-4o-mini (cheap)


## 📊 Part 9: Observability — Log Every Single Call

In production, you **must** log every LLM call: prompt, response, latency, cost, user_id, etc.

LiteLLM supports custom callbacks — here's a simple logger:

In [34]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[]


## 🔗 Part 10: Integrating the Gateway with LangChain

Here's where it really clicks for production GenAI apps:

**LangChain** for the orchestration (agents, chains, RAG) + **LiteLLM** as the unified LLM backend.

LangChain has a built-in `ChatLiteLLM` wrapper — drop it in like any other chat model.

In [27]:
%pip install -q langchain-litellm

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.4.0 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.9 which is incompatible.
langchain-groq 0.3.7 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.4.0 which is incompatible.
langchain-openai 0.3.32 requires langchain-core<1.0.0,>=0.3.74, but you have langchain-core 1.4.0 which is incompatible.
langchain-openai 0.3.32 requires openai<2.0.0,>=1.99.9, but you have openai 2.38.0 which is incompatible.
langchain-text-splitters 0.3.9 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.4.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe 

In [35]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

- **Definition**: An LLM Gateway is an interface or platform that facilitates interaction between users and large language models (LLMs), enabling seamless access to their capabilities.

- **Functionality**: It allows users to input queries and receive responses from LLMs, often incorporating features like API access, user authentication, and data management.

- **Applications**: LLM Gateways are used in various domains, including customer support, content generation, and educational tools, enhancing user experience and productivity.


## 🤖 Part 11: A Real Example — Multi-Provider LangChain Chain with Fallbacks

Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [36]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="gpt-4o-mini", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

18:27:48 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_latency.py", line 55, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:27:48 - LiteLLM:ERROR: lowest_cost.py:100 - litellm.router_strategy.lowest_cost.py::log_success_event(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_cost.py", line 35, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'


{
  "answer": {
    "benefits": [
      {
        "title": "Scalability",
        "description": "An LLM Gateway allows for the seamless scaling of language model resources, enabling organizations to handle varying workloads efficiently without compromising performance."
      },
      {
        "title": "Centralized Management",
        "description": "It provides a centralized interface for managing multiple language models, making it easier to deploy, monitor, and update models across different applications."
      },
      {
        "title": "Enhanced Security",
        "description": "An LLM Gateway can implement security protocols and access controls, ensuring that sensitive data is protected while interacting with language models."
      }
    ]
  }
}


## 🧪 Part 13: A Mini End-to-End Demo — Smart Router for a Chatbot

Let's build a tiny **task-aware chatbot** that:

1. Decides what kind of question the user is asking (code, summary, general)
2. Routes to the right model accordingly
3. Falls back if the chosen model fails
4. Logs cost and latency

In [37]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error


def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["gpt-4o",                     "gpt-4o-mini",   "groq/llama-3.3-70b-versatile"],
        "summary": ["gpt-4o-mini",                "groq/llama-3.3-70b-versatile"],
        "general": ["groq/llama-3.3-70b-versatile", "gpt-4o-mini"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }


# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

18:28:43 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_latency.py", line 55, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:43 - LiteLLM:ERROR: lowest_cost.py:100 - litellm.router_strategy.lowest_cost.py::log_success_event(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_cost.py", line 35, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'


❓ Q: Write a Python function to compute Fibonacci numbers.


18:28:45 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_latency.py", line 55, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:45 - LiteLLM:ERROR: lowest_cost.py:100 - litellm.router_strategy.lowest_cost.py::log_success_event(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_cost.py", line 35, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:45 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(

🏷️  Task:    code
🤖 Model:    gpt-4o-2024-08-06
⏱️  Latency: 2.51s
💰 Cost:    $0.003260
💬 Answer:  Certainly! The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding ones, typically starting with 0 and 1. Here is a Python function that computes Fibonacci numb...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.


18:28:46 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_latency.py", line 55, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:46 - LiteLLM:ERROR: lowest_cost.py:100 - litellm.router_strategy.lowest_cost.py::log_success_event(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_cost.py", line 35, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:46 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(

🏷️  Task:    summary
🤖 Model:    gpt-4o-mini-2024-07-18
⏱️  Latency: 0.88s
💰 Cost:    $0.000039
💬 Answer:  The attention mechanism allows neural networks to focus on specific parts of the input data, enhancing their ability to process and generate information by weighing the relevance of different elements...
❓ Q: Tell me a fun fact about elephants.
🏷️  Task:    general
🤖 Model:    llama-3.3-70b-versatile
⏱️  Latency: 0.6s
💰 Cost:    n/a
💬 Answer:  One fun fact about elephants is that they have a highly developed sense of empathy and cooperation. They have been known to display signs of grief, compassion, and even selflessness. For example, if a...


18:28:47 - LiteLLM:ERROR: lowest_latency.py:185 - litellm.proxy.hooks.prompt_injection_detection.py::async_pre_call_hook(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_latency.py", line 55, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
18:28:47 - LiteLLM:ERROR: lowest_cost.py:100 - litellm.router_strategy.lowest_cost.py::log_success_event(): Exception occured - 'NoneType' object has no attribute 'get'
Traceback (most recent call last):
  File "d:\Life\ML Hands-on\Github\AI-Agents-Langgraph-Concept\.venv\Lib\site-packages\litellm\router_strategy\lowest_cost.py", line 35, in log_success_event
    return
AttributeError: 'NoneType' object has no attribute 'get'
